# Import data from AEMO

In [0]:
# Systematically download aemo data
import os 
import requests
import time

OUT_DIR = "/Volumes/workspace/endava/aemo"
os.makedirs(OUT_DIR, exist_ok = True)
FILE_EXT = ".csv"

URL_TEMPLATE = "https://www.aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_{year}{month:02d}_{state}1.csv"

STATE = "QLD"
START_YEAR = 2006
END_YEAR = 2025

YEARS = range(START_YEAR, END_YEAR + 1)
MONTHS = range(1, 13)

for year in YEARS:
    for month in MONTHS:
        url = URL_TEMPLATE.format(state = STATE, year = year, month = month)
        filename = f"{OUT_DIR}/{STATE}-{year}-{month:02d}{FILE_EXT}"
        if os.path.exists(filename):
            print(f"Skipping {filename} (already exists)")
            continue

        try:
            print(f"Downloading {url} to {filename}")
            response = requests.get(url, timeout = 60)
            if response.status_code == 404:
                print(f"Not found: {url}")
                continue
            response.raise_for_status()
            with open(filename, "wb") as f:
                f.write(response.content)
            time.sleep(0.3)
        except Exception as e:
            print(f"Failed {url}: {e}")

print(f"\nAll files saved to: {OUT_DIR}")

In [0]:
# Investigate the first subset of files
import glob, os
import pandas as pd

DATA_DIR = "/Volumes/workspace/endava/aemo"
files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
files.sort()

# Test out with a manageable subset first including last 12 files
sample = pd.concat(
    (pd.read_csv(f) for f in files[-12:]),
    ignore_index = True
)

# AEMO columns often include: REGION, SETTLEMENTDATE, TOTALDEMAND, RRP, PERIODTYPE
sample["SETTLEMENTDATE"] = pd.to_datetime(sample["SETTLEMENTDATE"], errors = "coerce")
for col in ["TOTALDEMAND", "RRP"]:
    sample[col] = pd.to_numeric(sample[col], errors = "coerce")
    
for col in ["REGION", "PERIODTYPE"]:
    sample[col] = sample[col].astype("category")

print(sample.head())
print(sample.dtypes)
print(sample.describe(include = "all"))

In [0]:
# Try it for all files
df = pd.concat((pd.read_csv(f) for f in files), ignore_index = True)

# Cleaning
df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"], errors = "coerce")
for col in ["TOTALDEMAND", "RRP"]:
    df[col] = pd.to_numeric(df[col], errors = "coerce")
for col in ["REGION", "PERIODTYPE"]:
    df[col] = df[col].astype("category")

In [0]:
# Check whether our tables include NaN values or not
print(df.isna().sum())

## Preprocessing data

In [0]:
df_test = df.copy()

required_columns = {'SETTLEMENTDATE', 'REGION', 'TOTALDEMAND'}
missing = required_columns - set(df_test.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

df_test = df_test.set_index('SETTLEMENTDATE')[['TOTALDEMAND']].apply(pd.to_numeric, errors = 'coerce')

df_test.head()

We have 30-minute intervals before 01-10-2021 and after that are 5-minute intervals. We need to upsample the 30-minute era, which I proposed 2 solutions:
- "step": repeat each 30-min value across the six 5-min slots
- "linear": linearly interpolate between 30-min points
We look at both options as a hyperparameter to later see which one works better.

In [0]:
from typing import Literal, Optional

def enforce_5min_index(
  df: pd.DataFrame,
  mode: Literal['step', 'linear'] = 'linear',
  *,
  small_gap_limit: int = 3 # interpolate up to this many consecutive 5-min gaps
) -> pd.DataFrame:
  
  if df.empty:
    raise ValueError('Input dataframe is empty.')

  # Build a continuous 5-min index over the full span
  full_index = pd.date_range(
    df.index.min().floor('5min'),
    df.index.max().ceil('5min'),
    freq='5min'
  )

  s = df['TOTALDEMAND']

  # Reindex to union, then fill according to mode
  staged = s.reindex(s.index.union(full_index)).sort_index()

  if mode == 'step':
    filled = staged.ffill().reindex(full_index)

  elif mode == 'linear':
    filled = staged.interpolate(method='time', limit_direction = 'both').reindex(full_index)

  else:
    raise ValueError("mode must be 'step' or 'linear'.")

  # Tiny-gap smoothing (handles little glitches in the source if there are)
  filled = pd.to_numeric(filled, errors = "coerce")
  if small_gap_limit and small_gap_limit > 0:
    filled = filled.interpolate(method='time', limit = small_gap_limit)

  # Guards & energy'
  filled = filled.clip(lower=0)
  out = pd.DataFrame({
    "TOTALDEMAND": filled,
    "ENERGY_MWH": filled * (5.0 / 60.0)
  })

  return out

In [0]:
demand_linear_5min = enforce_5min_index(df_test, mode = "linear")

In [0]:
demand_linear_5min

In [0]:
# Sanity check on the target series
def get_target_series(df: pd.DataFrame, col = 'TOTALDEMAND') -> pd.Series:
    if col not in df.columns:
        raise ValueError(f"Expected column '{col}' in dataframe.")
    s = df[col].copy()

    # Ensure datetime index & sorted
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.copy()
        df.index = pd.to_datetime(df.index, errors = "raise")
        s = df[col]
    s = s.sort_index()

    # enforce 5-min freq
    s = s.asfreq("5min")

    # drop negative values
    s = pd.to_numeric(s, errors = "coerce").clip(lower = 0)
    return s

series = get_target_series(demand_linear_5min, col = 'TOTALDEMAND')

# Train / test split (we use the last 1 year as test set)
last_time = series.index.max()
test_start = (last_time - pd.DateOffset(years = 1)).ceil("5min")

train = series.loc[:test_start - pd.Timedelta(minutes = 5)].dropna()
test = series.loc[test_start:].dropna()

print("Train:", train.index.min(), " to ", train.index.max(), "| rows:", len(train))
print("Test:", test.index.min(), " to ", test.index.max(), "| rows:", len(test))

# Sanity check
def quick_check_info(s: pd.Series, name: str):
    print(f"\n{name} profile")
    print(" Missing:", int(s.isna().sum()))
    print(" Mean/Std:", round(s.mean(), 2), round(s.std(ddof = 1), 2))
    for q in (0.1, 0.5, 0.9):
        print(f" Q{q*100:.0f}:", round(s.quantile(q), 2))

quick_check_info(train, "Train")
quick_check_info(test, "Test")
    

## Test a few ML models

We will test with SARIMAX model first -

In [0]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Fit SARIMAX (daily seasonality at 5-min cadence = 288)
season_len = 288
model = SARIMAX(
  train,
  order = (2,0,2),
  seasonal_order=(1,1,1,season_len),
  enforce_stationarity = False,
  enforce_invertibility = False
)

res_sarimax = model.fit(disp = False)

# forecast the last year
forecast_sarimax = pd.Series(res_sarimax.forecast(steps = len(test)), index = test.index, name="SARIMAX_FORECAST")

# Generate synthetic data via bootstrap
# 1. Get in-sample one-step-ahead residuals (aligned to train)
insample_pred = pd.Series(res_sarimax.get_prediction().predicted_mean, index = train.index)
residuals = (train - insample_pred).dropna()

# 2. Bootstrap residuals and add to multi-step forecase to inject realistic variability
rng = np.random.default_rng(100)
boot = pd.Series(rng.choice(residuals.values, size = len(test), replace = true), index = test.index)

synthetic = pd.Series(
  np.maximum(0.0, forecast_sarimax.values + boot.values),
  index = test.index,
  name = "SARIMAX_SYNTHETIC"
)

In [0]:
# Evaluation forecast function
def evaluate_forecast(actual: pd.Series, predicted: pd.Series) -> pd.Series:
    err = predicted - actual
    mae  = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    mape = float(np.mean(np.abs(err / actual.replace(0, np.nan))) * 100.0)
    return pd.Series({
        "RMSE": rmse, "MAE": mae, "MAPE%": mape,
        "MeanDiff": float(predicted.mean() - actual.mean()),
        "StdRatio": float(predicted.std(ddof=1) / (actual.std(ddof=1) + 1e-9)),
    })

# Metrics on forecast
metrics = evaluate_forecast(test, forecast)
print("SARIMAX forecast metrics:\n", metrics)

# --- quick distributional check synthetic vs actual ---
summary = pd.concat({
    "Actual": test.describe(percentiles=[.1,.5,.9,.99])[["mean","std","min","max"]],
    "Synthetic": synthetic.describe(percentiles=[.1,.5,.9,.99])[["mean","std","min","max"]],
}, axis=1)
print("\nDistribution summary (Actual vs Synthetic):\n", summary)